In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys
current_path = os.getcwd() #获取你当前所在的文件夹路径
spotify_dab_path = os.path.abspath(os.path.join(current_path, "..", "..")) #：从当前子目录跳回项目根目录 spotify_dab
print(spotify_dab_path)

sys.path.append(spotify_dab_path)
from utils.transformations import reusable

/Workspace/Users/zzxhot123@outlook.com/SpotifyAzureProject/spotify_dab


## **DimUser**

In [0]:
df = spark.read \
    .format("parquet") \
    .load("abfss://bronze@zzxazureproject.dfs.core.windows.net/DimUser")

In [0]:
display(df)

user_id,user_name,country,subscription_type,start_date,end_date,updated_at
1,Carlos Berry,Switzerland,Premium,2023-10-17,null,2025-09-23T19:49:55.000Z
2,Amanda Jenkins,Montserrat,Family,2024-09-28,null,2025-09-29T19:49:55.000Z
3,Daniel Cook,Chile,Premium,2025-07-07,null,2025-09-08T19:49:55.000Z
4,Peter Hernandez,Nigeria,Free,2024-07-16,null,2025-09-29T19:49:55.000Z
5,Yolanda Morris,Aruba,Premium,2025-05-12,null,2025-09-17T19:49:55.000Z
6,Stephen Murphy,Cook Islands,Free,2025-06-17,null,2025-09-12T19:49:55.000Z
7,Anthony Andrews Jr.,Libyan Arab Jamahiriya,Free,2024-06-08,null,2025-09-20T19:49:55.000Z
8,Felicia Jones,Haiti,Family,2024-09-20,null,2025-09-08T19:49:55.000Z
9,Hector Decker,Palau,Family,2024-01-08,null,2025-09-29T19:49:55.000Z
10,Frank Davis,New Zealand,Family,2023-12-13,null,2025-09-22T19:49:55.000Z


## **Autoloader**

In [0]:
# 使用 Auto Loader 持续增量读取 DimUser
# 我要持续、自动、增量读取 bronze 里的 DimUser 数据，新文件来了自动读，结构变了自动适应，永不重复处理。
df_user = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimUser/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@zzxazureproject.dfs.core.windows.net/DimUser")

In [0]:
query_user = (df_user.writeStream
    .format("memory")
    .queryName("df_user_preview")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimUser/checkpoint_display")
    .start())
query_user.awaitTermination()
display(spark.sql("SELECT * FROM df_user_preview LIMIT 100"))

user_id,user_name,country,subscription_type,start_date,end_date,updated_at,_rescued_data
1,Carlos Berry,Switzerland,Premium,2023-10-17,null,2025-09-23T19:49:55.000Z,null
2,Amanda Jenkins,Montserrat,Family,2024-09-28,null,2025-09-29T19:49:55.000Z,null
3,Daniel Cook,Chile,Premium,2025-07-07,null,2025-09-08T19:49:55.000Z,null
4,Peter Hernandez,Nigeria,Free,2024-07-16,null,2025-09-29T19:49:55.000Z,null
5,Yolanda Morris,Aruba,Premium,2025-05-12,null,2025-09-17T19:49:55.000Z,null
6,Stephen Murphy,Cook Islands,Free,2025-06-17,null,2025-09-12T19:49:55.000Z,null
7,Anthony Andrews Jr.,Libyan Arab Jamahiriya,Free,2024-06-08,null,2025-09-20T19:49:55.000Z,null
8,Felicia Jones,Haiti,Family,2024-09-20,null,2025-09-08T19:49:55.000Z,null
9,Hector Decker,Palau,Family,2024-01-08,null,2025-09-29T19:49:55.000Z,null
10,Frank Davis,New Zealand,Family,2023-12-13,null,2025-09-22T19:49:55.000Z,null


In [0]:
df_transformed = df_user.withColumn("user_name" , upper(col("user_name")))

In [0]:
df_user_obj = reusable()

df_transformed = df_user_obj.dropColumns(df_transformed, ['_rescued_data'])
df_transformed = df_transformed.dropDuplicates(['user_id'])


In [0]:
query_transformed = (df_transformed.writeStream
    .format("memory")
    .queryName("df_transformed_preview")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimUser/checkpoint_display_transformed_cell10")
    .start())
query_transformed.awaitTermination()
display(spark.sql("SELECT * FROM df_transformed_preview LIMIT 100"))

user_id,user_name,country,subscription_type,start_date,end_date,updated_at
148,CHASE MOORE,Zambia,Family,2024-03-13,null,2025-09-18T19:49:55.000Z
463,JESSICA LEWIS,Slovenia,Free,2025-03-08,null,2025-10-06T19:49:55.000Z
471,JOSHUA BUTLER,Ethiopia,Free,2025-09-14,null,2025-09-25T19:49:55.000Z
496,ANDREW MATTHEWS,Taiwan,Family,2025-08-08,null,2025-09-17T19:49:55.000Z
243,MARTHA MARTIN,Chad,Premium,2024-05-11,null,2025-10-06T19:49:55.000Z
392,JENNIFER VILLANUEVA,Serbia,Free,2025-02-14,null,2025-10-06T19:49:55.000Z
31,TAMMY COOK,Bolivia,Premium,2025-02-15,null,2025-10-02T19:49:55.000Z
85,LISA SHERMAN,Japan,Family,2024-01-31,null,2025-09-11T19:49:55.000Z
137,AMY WEEKS,Sao Tome and Principe,Family,2024-07-02,null,2025-09-09T19:49:55.000Z
251,JERRY BOOTH,Lithuania,Family,2023-11-04,null,2025-09-17T19:49:55.000Z


In [0]:
(df_transformed.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimUser/checkpoint")
    .trigger(once=True)
    .option("path", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_cata.silver.DimUser"))

## **DimArtist**


In [0]:
df_art = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimArtist/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@zzxazureproject.dfs.core.windows.net/DimArtist")

In [0]:
query_user = (df_art.writeStream
    .format("memory")
    .queryName("df_artist_preview")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimArtist/checkpoint_display")
    .start())
query_user.awaitTermination()
display(spark.sql("SELECT * FROM df_artist_preview LIMIT 100"))

artist_id,artist_name,genre,country,updated_at,_rescued_data
1,Lauren Singleton,Rock,Oman,2025-09-26T19:49:55.000Z,null
2,Gregory Mccarty,Classical,Suriname,2025-09-23T19:49:55.000Z,null
3,Adam Stanton,Hip-Hop,Anguilla,2025-09-09T19:49:55.000Z,null
4,Monica Roth,Jazz,Argentina,2025-09-15T19:49:55.000Z,null
5,Calvin Blake,Rock,Greenland,2025-09-18T19:49:55.000Z,null
6,James Lowe,Pop,Saudi Arabia,2025-09-17T19:49:55.000Z,null
7,Jacqueline Evans,Jazz,Syrian Arab Republic,2025-09-10T19:49:55.000Z,null
8,Paul Johnson,Electronic,Bangladesh,2025-09-19T19:49:55.000Z,null
9,Alexa Garcia,Rock,Burkina Faso,2025-09-09T19:49:55.000Z,null
10,Robert Ellis,Rock,Ecuador,2025-09-19T19:49:55.000Z,null


In [0]:
df_art_obj = reusable()

df_art = df_art_obj.dropColumns(df_art, ['_rescued_data'])
df_art = df_art.dropDuplicates(['artist_id'])


In [0]:
(df_art.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimArtist/checkpoint")
    .trigger(once=True)
    .option("path", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_cata.silver.DimArtist"))

## **DimTrack**


In [0]:
df_tra = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimTrack/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@zzxazureproject.dfs.core.windows.net/DimTrack")


query_user = (df_tra.writeStream
    .format("memory")
    .queryName("df_tra_preview")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimTrack/checkpoint_display")
    .start())
query_user.awaitTermination()
display(spark.sql("SELECT * FROM df_tra_preview LIMIT 100"))

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,_rescued_data
1,Stand-alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z,null
2,Programmable contextually-based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z,null
3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z,null
4,Mandatory user-facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z,null
5,Multi-layered needs-based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,null
6,Synchronized high-level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z,null
7,Total human-resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z,null
8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z,null
9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z,null
10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z,null


In [0]:
df_tra = df_tra.withColumn("durationFlag" , when(col('duration_sec')<=150, "low")\
    .when(col('duration_sec')<=300, "medium")\
    .otherwise("high"))

df_tra = df_tra.withColumn("track_name", regexp_replace(col("track_name"),'-', ' '))

df_tra_obj = reusable()

df_tra = df_tra_obj.dropColumns(df_tra, ['_rescued_data'])
df_tra = df_tra.dropDuplicates(['track_id'])



In [0]:

query_user = (df_tra.writeStream
    .format("memory")
    .queryName("df_tra_preview")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimTrack/checkpoint_display_cell20")
    .start())
query_user.awaitTermination()
display(spark.sql("SELECT * FROM df_tra_preview LIMIT 100"))

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,durationFlag
148,Synchronized cohesive monitoring,422,People Album,168,2021-10-19,2025-09-24T19:49:55.000Z,medium
463,Function based foreground time frame,456,Where Album,142,2024-03-04,2025-09-16T19:49:55.000Z,low
471,Right sized demand driven moderator,440,Imagine Album,230,2021-06-20,2025-09-20T19:49:55.000Z,medium
496,Reverse engineered human resource encryption,142,Right Album,359,2025-06-23,2025-09-09T19:49:55.000Z,high
243,Compatible mission critical contingency,452,Keep Album,200,2023-09-19,2025-09-24T19:49:55.000Z,medium
392,Self enabling multi state success,13,Around Album,316,2023-07-30,2025-09-21T19:49:55.000Z,high
31,Expanded tertiary firmware,154,Country Album,323,2021-09-02,2025-09-20T19:49:55.000Z,high
85,Optional fault tolerant time frame,342,Painting Album,161,2025-04-17,2025-09-21T19:49:55.000Z,medium
137,Enhanced 6thgeneration task force,226,Religious Album,95,2021-12-09,2025-09-30T19:49:55.000Z,low
251,Synergistic optimizing installation,221,Both Album,92,2021-04-02,2025-09-10T19:49:55.000Z,low


In [0]:
(df_tra.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimTrack/checkpoint")
    .trigger(once=True)
    .option("path", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_cata.silver.DimTrack"))

## **DimDate**


In [0]:
df_dat = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimDate/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@zzxazureproject.dfs.core.windows.net/DimDate")

In [0]:
df_dat_obj = reusable()

df_dat = df_dat_obj.dropColumns(df_dat, ['_rescued_data'])



In [0]:
(df_dat.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimDate/checkpoint")
    .trigger(once=True)
    .option("path", "abfss://silver@zzxazureproject.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_cata.silver.DimDate"))

## **FactStream**


In [0]:
df_fact = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/FactStream/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@zzxazureproject.dfs.core.windows.net/FactStream")

query_user = (df_fact.writeStream
    .format("memory")
    .queryName("df_fact_preview")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/FactStream/checkpoint_display")
    .start())
query_user.awaitTermination()
display(spark.sql("SELECT * FROM df_fact_preview LIMIT 100"))

stream_id,user_id,track_id,date_key,listen_duration,device_type,stream_timestamp,_rescued_data
1,361,74,20250518,156,Smart Speaker,2025-09-30T19:49:55.000Z,null
2,321,288,20250519,47,Mobile,2025-09-27T19:49:55.000Z,null
3,275,340,20250307,214,Mobile,2025-10-03T19:49:55.000Z,null
4,43,373,20250216,14,Desktop,2025-10-04T19:49:55.000Z,null
5,319,95,20250421,266,Desktop,2025-09-27T19:49:55.000Z,null
6,52,31,20250130,317,Desktop,2025-10-02T19:49:55.000Z,null
7,5,354,20250824,90,Mobile,2025-10-01T19:49:55.000Z,null
8,115,386,20250413,159,Smart Speaker,2025-09-29T19:49:55.000Z,null
9,439,95,20250930,290,Mobile,2025-10-04T19:49:55.000Z,null
10,40,389,20250227,53,Mobile,2025-10-01T19:49:55.000Z,null


In [0]:
df_fact_obj = reusable()

df_fact = df_fact_obj.dropColumns(df_fact, ['_rescued_data'])



In [0]:
(df_fact.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@zzxazureproject.dfs.core.windows.net/FactStream/checkpoint")
    .trigger(once=True)
    .option("path", "abfss://silver@zzxazureproject.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_cata.silver.FactStream"))